# liar_trace

> Phase 15 (`specs/multiagent/phase15/`): clue-board runs traced turn by turn. `Tracer` records, before each honest agent acts, the clues it knows for its open subtask (with their truth) and how many open subtasks show a false post; `traceRun` runs a cell exactly as `runClueNumpy` / `runClue` do with the tracer around each honest member; `cacheOnlyClient` replays Haiku cells from the response cache and raises on a miss; `VariantAgent` is the Phase 11 rule agent with one candidate mechanism switched on (`relay`, `first`, `repost`).

In [ ]:
#| default_exp liar_trace

In [ ]:
#| hide
from nbdev.showdoc import *

## The tracer

`known(...)` is pure and every member starts its turn with `world.advance(t)`, so the tracer's own `advance(t)` and `known` calls leave the run unchanged. Liars are not wrapped. A trace entry: `(t, i, k, knownK, contam, falseVisible, nOpen)` with `knownK = {j: (r, true?)}` for the agent's open subtask `k` (None when it has none).

In [ ]:
#| export
import multiprocessing as mp
from collections import namedtuple
from concurrent.futures import ProcessPoolExecutor

from explRL.bench_grid import rowKey
from explRL.clue_agents import ClueBoard, ClueLiar, RuleAgent, candidates, known
from explRL.clue_fix import fixOptions
from explRL.clue_runs import clueMetrics, clueSetup
from explRL.clue_text import TextClueEnv
from explRL.llm_agent import LLMMember, runTurns
from explRL.llm_budget import estimateCost
from explRL.llm_client import LLMClient, ResponseCache
from explRL.llm_fake import FakeClient
from explRL.llm_runs import LLMConfig, cellRow, llmStats, makeClient
from explRL.multi import spawnRngs

TraceRun = namedtuple('TraceRun', 'rows log trace board nLiar T')
Entry = namedtuple('Entry', 't i k knownK contam falseVisible nOpen')
VARIANTS = (None, 'relay', 'first', 'repost')


def falseOpen(b, t):
    "Open subtasks with a false post at or before t."
    return {s[2] for s in b.shareLog if s[6] and s[0] <= t and s[2] in b.open}


class Tracer:
    "Wraps one honest member: records an `Entry` before it acts; everything else (the `id` runTurns sets too) goes to it."
    def __init__(self, member, channel, out):
        self.__dict__.update(member=member, channel=channel, out=out)

    def __getattr__(self, a):
        return getattr(self.__dict__['member'], a)

    def __setattr__(self, a, v):
        setattr(self.member, a, v)

    def event(self, world, t):
        world.advance(t)
        m, i = self.member, self.member.id
        kn = m.knownFor(world) if hasattr(m, 'knownFor') else known(i, world, self.channel)
        k = world.opened[i]
        kk = {j: (r, world.tasks[k].task.clue(j)[1] == r) for j, r in kn[k].items()} if k in world.open else None
        self.out.append(Entry(float(t), i, k, kk, bool(kk) and not all(ok for _, ok in kk.values()),
                              len(falseOpen(world, t)), len(world.open)))
        return m.event(world, t)

## Rule-agent variants

`VariantAgent(variant=None)` is `RuleAgent` unchanged. `relay`: after its own clues, a clue it knows from the board (not held), for a subtask a teammate has open, that it has not posted itself. `first`: the earliest sighting of a clue wins (own clues still win). `repost`: its own slot counts as shared only when some post of it carries the true (its own) residue.

In [ ]:
#| export
def knownFirst(i, b, channel):
    "known() on the blackboard with the earliest sighting winning; own clues still win."
    if not isinstance(channel, ClueBoard):
        raise ValueError("the 'first' variant needs the blackboard")
    out = {k: {} for k in b.open}
    for k, xs in channel.posts.items():
        for x in sorted(xs, key=lambda x: x[4]):
            if k in out and x[0] not in out[k]:
                out[k][x[0]] = x[2]
    for k in out:
        out[k].update(b.held(i, k))
    return out


class VariantAgent(RuleAgent):
    "The Phase 11 rule agent with one candidate mechanism of Phase 15 switched on (variant None: RuleAgent unchanged)."
    def __init__(self, channel, rng, share='needed', guess=0.0, variant=None):
        if variant not in VARIANTS:
            raise ValueError(f'unknown variant {variant}')
        super().__init__(channel, rng, share, guess)
        self.variant = variant

    def knownFor(self, b):
        return knownFirst(self.id, b, self.channel) if self.variant == 'first' else known(self.id, b, self.channel)

    def _shareable(self, b, kn, t):
        if self.variant not in ('relay', 'repost') or self.channel is None or self.share == 'never':
            return super()._shareable(b, kn, t)
        others = {o for a, o in enumerate(b.opened) if a != self.id}
        if self.variant == 'repost':
            done = {(s[2], s[3]) for s in b.shareLog if not s[6]}
            for k in sorted(kn):
                if self.share == 'needed' and k not in others:
                    continue
                for j, r in sorted(b.held(self.id, k).items()):
                    if (k, j) not in done:
                        return ('share', (k, j, r))
            return None
        if act := super()._shareable(b, kn, t):
            return act
        mine = {(s[2], s[3], s[4]) for s in b.shareLog if s[1] == self.id}
        for k in sorted(kn):
            if k in others:
                held = b.held(self.id, k)
                for j, r in sorted(kn[k].items()):
                    if j not in held and (k, j, r) not in mine:
                        return ('share', (k, j, r))
        return None

    def choose(self, b, t):
        if self.variant != 'first':
            return super().choose(b, t)
        kn, k = self.knownFor(b), b.opened[self.id]       # RuleAgent.choose with this agent's own known clues
        if k in b.open and len(c := candidates(b, k, kn[k])) == 1:
            b.noteSubmit(self.id, 1)
            return ('submit', c[0])
        if act := self._shareable(b, kn, t):
            return act
        live = [x for x in b.open if b.tasks[x].expire > t + 1] or list(b.open)
        best = max(len(kn[x]) for x in live)
        top = [x for x in live if len(kn[x]) == best]
        top = [x for x in top if b.tasks[x].value == max(b.tasks[y].value for y in top)]
        if k not in top:
            return ('open', int(top[int(self.rng.integers(len(top)))]))
        if self.rng.random() < self.guess:
            c = candidates(b, k, kn[k])
            b.noteSubmit(self.id, len(c))
            return ('submit', int(c[int(self.rng.integers(len(c)))]))
        return ('wait', None)

## Traced runs and the cache-only client

In [ ]:
#| export
class CacheMiss(RuntimeError):
    "A request the response cache does not hold (the cache-only client never calls the API)."


def _miss(req):
    raise CacheMiss('cache miss: this request was never made live')


def cacheOnlyClient(cfg, cacheDir):
    "The Phase 10 client over the response cache with an offline stand-in for the API that raises on any call."
    return LLMClient(cfg.model, cfg.effort, cfg.maxTokens, ResponseCache(cacheDir), None, client=FakeClient(_miss))


def traceRun(cfg, seed, client=None, variant=None):
    """One clue-board cell exactly as runClueNumpy (client None) or runClue (client given), with a Tracer around each
    honest member -> TraceRun(rows, log, trace, board, nLiar, T). variant: numpy honest agents only."""
    if any(fixOptions(cfg.world).values()):
        raise ValueError('Phase 15 traces baseline cells only (no Phase 13-14 options)')
    if client is not None and variant is not None:
        raise ValueError('variants apply to rule agents only')
    T, nL, trace = cfg.turnsPerAgent(), cfg.nLiar, []
    b, ch = clueSetup(cfg, seed)
    if client is None:
        rngs = spawnRngs(seed + 1000, cfg.N)
        members = [ClueLiar(ch, rngs[i]) if i < nL else
                   VariantAgent(ch, rngs[i], cfg.world.get('share', 'needed'), cfg.world.get('guess', 0.0), variant)
                   for i in range(cfg.N)]
        llm = []
    else:
        client.reserve(estimateCost(cfg.model, 'clue', cfg.N - nL, T))
        env = TextClueEnv(b, cfg.world.get('hint'), cfg.world.get('showOpen', False), **fixOptions(cfg.world))
        llm = [LLMMember(client, env, ch, cfg.memory, cfg.framing) for _ in range(cfg.N - nL)]
        members = [ClueLiar(ch, r) for r in spawnRngs(seed + 3000, nL)] + llm
    hist = runTurns([m if i < nL else Tracer(m, ch, trace) for i, m in enumerate(members)], b, T)
    row = dict(episode=0, **clueMetrics(cfg, b, hist, range(nL, cfg.N), T), **(llmStats(llm) if client is not None else {}))
    return TraceRun([row], [dict(episode=0, **x) for m in llm for x in m.log], trace, b, nL, T)


def _traceCell(args):
    cell, cache = args
    cfg = LLMConfig.fromDict(cell['cfg'])
    client = None if cell.get('runner') == 'numpy' else cacheOnlyClient(cfg, cache) if cache else makeClient(cfg)
    tr = traceRun(cfg, cell['seed'], client, cell['cfg']['world'].get('variant'))
    rows = [cellRow(cell, cfg, r) for r in tr.rows]
    key = rowKey(rows[0])
    return key, tr._replace(rows=rows, log=[dict(key=key, **x) for x in tr.log])


def traceCells(cells, cache=None, workers=1):
    """{row key: TraceRun} for clue-board cells (rows as cellRow, logs keyed as runLLMCell's). LLM cells: the
    cache-only client over `cache`, else the fake client. A variant is read from cfg.world['variant']."""
    args = [(c, str(cache) if cache else None) for c in cells]
    if workers > 1:
        with ProcessPoolExecutor(workers, mp_context=mp.get_context('spawn')) as ex:
            return dict(ex.map(_traceCell, args))
    return dict(map(_traceCell, args))

## Tests

In [ ]:
import json
import numpy as np
from explRL.clue_agents import ClueBoard, RuleAgent, known
from explRL.clue_grid import CLUE_SETTINGS, _board, clueCell
from explRL.clue_runs import runClue, runClueNumpy
from explRL.clueboard import ClueBoardEnv
from explRL.fix_grid import fixCell
from explRL.liar_trace import *
from explRL.llm_runs import LLMConfig, makeClient

numpyCell = lambda N, nLiar, sd, turns=40, **w: clueCell('t', 'x', N, sd, 'numpy', channel='blackboard', turns=turns, nLiar=nLiar,
                                                          world=dict(_board(CLUE_SETTINGS), guess=0.0, share='needed', **w))
cfgOf = lambda c: LLMConfig.fromDict(c['cfg'])
# AC-2: traced numpy runs equal untraced ones, with and without liars; one entry per honest turn
for N, nL in ((4, 0), (8, 2), (16, 4)):
    for sd in (2000, 2001):
        cfg = cfgOf(numpyCell(N, nL, sd))
        tr = traceRun(cfg, sd)
        assert json.dumps(tr.rows, sort_keys=True) == json.dumps(runClueNumpy(cfg, sd), sort_keys=True), (N, nL, sd)
        assert len(tr.trace) == (N - nL) * 40 and {e.i for e in tr.trace} == set(range(nL, N)) and tr.nLiar == nL
# AC-2: traced LLM runs (fake client) equal runClue's rows and logs, clean and with liars (short runs)
for nL in (0, 2):
    c = fixCell('t', 'baseline', 8, 900, nL, turns=4)
    cfg = cfgOf(c)
    tr = traceRun(cfg, 900, makeClient(cfg))
    rows, log = runClue(cfg, 900, makeClient(cfg))
    assert json.dumps(tr.rows, sort_keys=True) == json.dumps(rows, sort_keys=True) and tr.log == log, nL
    assert len(tr.trace) == (8 - nL) * 4

In [ ]:
# AC-2: the trace's known clues equal known() at each turn: replay the numpy run step by step and compare
cfg = cfgOf(numpyCell(8, 2, 2003))
tr = traceRun(cfg, 2003)
from explRL.clue_runs import clueSetup
from explRL.multi import spawnRngs
from explRL.clue_agents import ClueLiar
b, ch = clueSetup(cfg, 2003)
rngs = spawnRngs(2003 + 1000, 8)
mem = [ClueLiar(ch, rngs[i]) if i < 2 else RuleAgent(ch, rngs[i], 'needed', 0.0) for i in range(8)]
seen = []
class Probe:
    def __init__(s, m): s.__dict__['m'] = m
    def __getattr__(s, a): return getattr(s.__dict__['m'], a)
    def __setattr__(s, a, v): setattr(s.m, a, v)
    def event(s, w, t):
        w.advance(t)
        k = w.opened[s.m.id]
        seen.append((float(t), s.m.id, k, known(s.m.id, w, ch)[k] if k in w.open else None))
        return s.m.event(w, t)
from explRL.llm_agent import runTurns
runTurns([m if i < 2 else Probe(m) for i, m in enumerate(mem)], b, 40)
assert [(e.t, e.i, e.k, None if e.knownK is None else {j: r for j, (r, _) in e.knownK.items()}) for e in tr.trace] == seen
# contamination is flagged exactly when a known residue is false; some turns see false posts
assert all(e.contam == (e.knownK is not None and any(not ok for _, ok in e.knownK.values())) for e in tr.trace)
assert any(e.falseVisible > 0 for e in tr.trace) and all(0 <= e.falseVisible <= e.nOpen for e in tr.trace)

In [ ]:
# the cache-only client: replays a cached reply, raises on a miss, never needs a key or anthropic
import tempfile
from explRL.llm_client import ResponseCache, requestKey
cfg = cfgOf(fixCell('t', 'baseline', 4, 900, turns=2))
with tempfile.TemporaryDirectory() as d:
    fake = makeClient(cfg, dict(kind='fake', cache=d))
    rows, log = runClue(cfg, 900, fake)                  # fills the cache with fake replies
    tr = traceRun(cfg, 900, cacheOnlyClient(cfg, d))     # replays them exactly
    assert tr.log == log and json.dumps(tr.rows, sort_keys=True) == json.dumps(rows, sort_keys=True)
    try:
        traceRun(cfgOf(fixCell('t', 'baseline', 4, 901, turns=2)), 901, cacheOnlyClient(cfg, d))
        raise AssertionError('a miss must raise')
    except CacheMiss:
        pass
# traceRun refuses Phase 13-14 options; variants only for rule agents
for bad in (lambda: traceRun(cfgOf(fixCell('t', 'record', 4, 900, True, turns=2)), 900),
            lambda: traceRun(cfg, 900, makeClient(cfg), 'relay')):
    try:
        bad()
        raise AssertionError
    except ValueError:
        pass

In [ ]:
# AC-5: VariantAgent(None) is RuleAgent unchanged (rows byte-identical on forecast-like cells)
for N, nL, sd in ((8, 2, 2000), (32, 8, 2001), (4, 0, 2002)):
    cfg = cfgOf(numpyCell(N, nL, sd))
    assert json.dumps(traceRun(cfg, sd).rows) == json.dumps(runClueNumpy(cfg, sd)), (N, nL, sd)
# each variant on a hand-made board: agent 1 holds T0's clue 0; the board shows a false T0 clue 0 then a true T0 clue 1
b = ClueBoardEnv(7, 3, m=2, r=1, R=30, value=(1, 2, 3), life=30, penalty=0.5)
k = b.open[0]
t0 = b.tasks[k].task
(j0, (p0, r0)), (j1, (p1, r1)) = [(j, t0.clue(j)) for j in range(2)]
holder0 = next(iter(b.tasks[k].holders[0]))
ch = ClueBoard()
ch.post(0, k, (j1, p1, (r1 + 1) % p1), 0.0)          # a false post of clue 1 first
ch.post(2, k, (j1, p1, r1), 1.0)                      # then the true one
b.shareLog += [(0.0, 0, k, j1, (r1 + 1) % p1, False, True), (1.0, 2, k, j1, r1, False, False)]
a = VariantAgent(ch, np.random.default_rng(0), variant='first'); a.id = holder0
assert a.knownFor(b)[k][j1] == (r1 + 1) % p1                                  # the first sighting (false) wins
a.variant = None
assert a.knownFor(b)[k][j1] == r1                                            # the latest (true) wins
# relay: the agent holding no clue of k, with a teammate on k, shares the board's clue 1 (latest sighting: true); RuleAgent does not
other = next(i for i in range(3) if not any(i in hs for hs in b.tasks[k].holders))
b.opened[(other + 1) % 3] = k
r = VariantAgent(ch, np.random.default_rng(0), variant='relay'); r.id = other
base = VariantAgent(ch, np.random.default_rng(0)); base.id = other
assert r._shareable(b, known(other, b, ch), 1.0) == ('share', (k, j1, r1))
assert base._shareable(b, known(other, b, ch), 1.0) is None
# repost: the holder of clue 1, whose slot only a liar posted (falsely), still shares it; RuleAgent does not
b2 = ClueBoardEnv(7, 3, m=2, r=1, R=30, value=(1, 2, 3), life=30, penalty=0.5)
h = next(iter(b2.tasks[k].holders[1]))
liar = next(i for i in range(3) if i != h)
b2.opened[liar] = k
b2.shareLog.append((0.0, liar, k, j1, (r1 + 1) % p1, False, True))
ch2 = ClueBoard(); ch2.post(liar, k, (j1, p1, (r1 + 1) % p1), 0.0)
rp = VariantAgent(ch2, np.random.default_rng(0), variant='repost'); rp.id = h
base = VariantAgent(ch2, np.random.default_rng(0)); base.id = h
kn = known(h, b2, ch2)
assert rp._shareable(b2, kn, 1.0) == ('share', (k, j1, r1)) and base._shareable(b2, kn, 1.0) is None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()